# Workflow All  - 4-stage Deep Learning Funnel

Notebook này điều khiển toàn bộ Phase 2 theo 4 stage, tối đa 35 training runs. Mỗi training cell chỉ chạy đúng một mô hình. Selector chỉ dùng validation metric; test metric chỉ dùng để báo cáo.

## Setup và helper functions

Chạy cell này trước. Notebook tự nhận khi chạy từ project root hoặc từ thư mục `Phần 2 mô hình`.

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import pandas as pd
import torch

PHASE2_DIR = Path.cwd()
if not (PHASE2_DIR / "train_gru.py").is_file():
    PHASE2_DIR = Path("Phần 2 mô hình").resolve()
if not (PHASE2_DIR / "train_gru.py").is_file():
    raise FileNotFoundError("Run this notebook from the project root or from 'Phần 2 mô hình'.")

sys.path.insert(0, str(PHASE2_DIR))

import train_gru
import train_attention_gru as train_attention
import train_multi_output_best_model as train_multi

GRU_RUNS_ROOT = PHASE2_DIR / "runs" / "gru_baseline"
ATTENTION_RUNS_ROOT = PHASE2_DIR / "runs" / "attention_gru"
MULTI_RUNS_ROOT = PHASE2_DIR / "runs" / "multi_output_best_model"
RUN_LOG: list[dict[str, object]] = []


def timed_train (stage: str, train_fn, config, **kwargs):
    start = time.perf_counter()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    metrics = train_fn(config, **kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_minutes = (time.perf_counter() - start) / 60.0
    metrics["elapsed_minutes"] = elapsed_minutes
    run_dir = Path(str(metrics["run_dir"]))
    metrics_path = run_dir / "metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    RUN_LOG.append({"stage": stage, "experiment_name": config.experiment_name, "elapsed_minutes": elapsed_minutes, "run_dir": str(run_dir)})
    print(f"{stage} | {config.experiment_name} | {elapsed_minutes:.2f} min")
    return metrics


def read_json(path: Path) -> dict[str, object]:
    return json.loads(path.read_text(encoding="utf-8"))


def is_completed_run(run_dir: Path) -> bool:
    metrics_path = run_dir / "metrics.json"
    config_path = run_dir / "config.json"
    best_model_path = run_dir / "best_model.pt"
    if not metrics_path.is_file() or not config_path.is_file() or not best_model_path.is_file():
        return False
    metrics = read_json(metrics_path)
    config = read_json(config_path)
    name = str(config.get("experiment_name", run_dir.name)).lower()
    if "smoke" in name or "smoke" in run_dir.name.lower():
        return False
    if config.get("max_train_batches") is not None or config.get("max_eval_batches") is not None:
        return False
    return int(metrics.get("epochs_ran", 0)) > 1


def load_planned_status(stage_name: str, planned_runs: list[dict[str, object]], runs_root: Path, metric_columns: list[str]) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for item in planned_runs:
        run_dir = runs_root / str(item["experiment"])
        metrics_path = run_dir / "metrics.json"
        row = {"stage": stage_name, **item, "status": "pending", "run_dir": str(run_dir)}
        if metrics_path.is_file():
            metrics = read_json(metrics_path)
            row["status"] = "completed" if is_completed_run(run_dir) else "incomplete_or_smoke"
            for column in metric_columns:
                row[column] = metrics.get(column)
        rows.append(row)
    return pd.DataFrame(rows)


def rank_completed(status: pd.DataFrame, metric: str, loss_metric: str | None = None, top_k: int = 5) -> pd.DataFrame:
    completed = status[status["status"] == "completed"].copy()
    if completed.empty or metric not in completed.columns:
        return completed
    completed[metric] = pd.to_numeric(completed[metric], errors="coerce")
    completed = completed.dropna(subset=[metric])
    sort_cols = [metric]
    ascending = [False]
    if loss_metric is not None and loss_metric in completed.columns:
        completed[loss_metric] = pd.to_numeric(completed[loss_metric], errors="coerce")
        sort_cols.append(loss_metric)
        ascending.append(True)
    sort_cols.append("run")
    ascending.append(True)
    return completed.sort_values(sort_cols, ascending=ascending).head(top_k)


def select_top_configs(planned_runs: list[dict[str, object]], runs_root: Path, metric: str, loss_metric: str | None, top_k: int) -> tuple[list[dict[str, object]], pd.DataFrame]:
    status = load_planned_status("selection", planned_runs, runs_root, [metric, loss_metric or ""])
    ranking = rank_completed(status, metric, loss_metric, top_k)
    selected: list[dict[str, object]] = []
    for _, row in ranking.iterrows():
        run_dir = Path(str(row["run_dir"]))
        config = read_json(run_dir / "config.json")
        selected.append({"run": row["run"], "experiment": row["experiment"], "metric": row.get(metric), "config": config, "run_dir": str(run_dir)})
    return selected, ranking


def fallback_config(**values):
    return values


def pick_config(bank: list[dict[str, object]], index: int, fallback: dict[str, object]) -> dict[str, object]:
    if len(bank) > index:
        return dict(bank[index]["config"])
    return dict(fallback)


def apply_common(config, selected: dict[str, object], keys: list[str]) -> None:
    for key in keys:
        if hasattr(config, key) and key in selected:
            setattr(config, key, selected[key])


def show_status() -> pd.DataFrame:
    frames = [
        load_planned_status("Stage 1 GRU", GRU_PLANNED, GRU_RUNS_ROOT, ["best_val_macro_f1", "best_val_loss", "test_macro_f1", "test_accuracy", "elapsed_minutes", "epochs_ran"]),
        load_planned_status("Stage 2 Attention", ATTENTION_PLANNED, ATTENTION_RUNS_ROOT, ["best_val_macro_f1", "best_val_loss", "test_macro_f1", "test_accuracy", "elapsed_minutes", "epochs_ran"]),
        load_planned_status("Stage 3 Multi", MULTI_PLANNED, MULTI_RUNS_ROOT, ["best_val_composite_score", "val_cell_masked_macro_f1", "test_cell_masked_macro_f1", "test_pose_macro_f1", "elapsed_minutes", "epochs_ran"]),
        load_planned_status("Stage 4 Attention Final", FINAL_ATTENTION_PLANNED, ATTENTION_RUNS_ROOT, ["best_val_macro_f1", "best_val_loss", "test_macro_f1", "elapsed_minutes", "epochs_ran"]),
        load_planned_status("Stage 4 Multi Final", FINAL_MULTI_PLANNED, MULTI_RUNS_ROOT, ["best_val_composite_score", "val_cell_masked_macro_f1", "test_cell_masked_macro_f1", "elapsed_minutes", "epochs_ran"]),
    ]
    return pd.concat(frames, ignore_index=True)


C:\Users\nguyen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Planned experiment matrix

Tên run dùng prefix `S1_/S2_/S3_/S4_` để không lẫn với các run cũ. Không xóa thư mục `runs/`; dashboard chỉ đọc planned runs.

In [2]:
GRU_PLANNED = [
    {"run": "S1_G01", "experiment": "S1_G01_raw_h64", "profile": "raw", "hidden_size": 64, "num_layers": 1, "dropout": 0.0, "batch_size": 32, "lr": 1e-3, "purpose": "raw baseline đối chứng"},
    {"run": "S1_G02", "experiment": "S1_G02_raw_shift_h64_anchor", "profile": "raw_shift", "hidden_size": 64, "num_layers": 1, "dropout": 0.0, "batch_size": 32, "lr": 1e-3, "purpose": "anchor ổn định từ G02"},
    {"run": "S1_G03", "experiment": "S1_G03_raw_shift_h96", "profile": "raw_shift", "hidden_size": 96, "num_layers": 1, "dropout": 0.0, "batch_size": 32, "lr": 1e-3, "purpose": "capacity trung bình"},
    {"run": "S1_G04", "experiment": "S1_G04_raw_shift_h128", "profile": "raw_shift", "hidden_size": 128, "num_layers": 1, "dropout": 0.0, "batch_size": 32, "lr": 1e-3, "purpose": "capacity cao kiểm tra overfit"},
    {"run": "S1_G05", "experiment": "S1_G05_raw_shift_h64_l2_d03", "profile": "raw_shift", "hidden_size": 64, "num_layers": 2, "dropout": 0.3, "batch_size": 32, "lr": 1e-3, "purpose": "stacked GRU nhỏ"},
    {"run": "S1_G06", "experiment": "S1_G06_raw_shift_h96_l2_d03", "profile": "raw_shift", "hidden_size": 96, "num_layers": 2, "dropout": 0.3, "batch_size": 32, "lr": 1e-3, "purpose": "stacked GRU trung bình"},
    {"run": "S1_G07", "experiment": "S1_G07_raw_shift_h96_b64_lr0005", "profile": "raw_shift", "hidden_size": 96, "num_layers": 1, "dropout": 0.0, "batch_size": 64, "lr": 5e-4, "purpose": "batch/lr ổn định"},
]

ATTENTION_PLANNED = [
    {"run": "S2_A01", "experiment": "S2_A01_top_gru_a32", "profile": "selected", "hidden_size": "selected", "attention_dim": 32, "num_layers": "selected", "dropout": 0.2, "lr": "selected", "purpose": "top GRU -> attention a32"},
    {"run": "S2_A02", "experiment": "S2_A02_top_gru_a48", "profile": "selected", "hidden_size": "selected", "attention_dim": 48, "num_layers": "selected", "dropout": 0.2, "lr": "selected", "purpose": "top GRU -> attention a48"},
    {"run": "S2_A03", "experiment": "S2_A03_second_gru_a32", "profile": "selected", "hidden_size": "selected", "attention_dim": 32, "num_layers": "selected", "dropout": 0.2, "lr": "selected", "purpose": "second GRU branch"},
    {"run": "S2_A04", "experiment": "S2_A04_second_gru_l2_d03", "profile": "selected", "hidden_size": "selected", "attention_dim": 32, "num_layers": 2, "dropout": 0.3, "lr": "selected", "purpose": "stacked attention branch"},
    {"run": "S2_A05", "experiment": "S2_A05_raw_shift_h128_a32_d03", "profile": "raw_shift", "hidden_size": 128, "attention_dim": 32, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "purpose": "h128 regularized"},
    {"run": "S2_A06", "experiment": "S2_A06_raw_shift_h96_a64_d03", "profile": "raw_shift", "hidden_size": 96, "attention_dim": 64, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "purpose": "attention width upper"},
    {"run": "S2_A07", "experiment": "S2_A07_phase_hampel_shift_h96_a32", "profile": "phase_hampel_shift", "hidden_size": 96, "attention_dim": 32, "num_layers": 1, "dropout": 0.2, "lr": 1e-3, "purpose": "denoise+shift stability"},
    {"run": "S2_A08", "experiment": "S2_A08_phase_hampel_shift_noise_lr0005", "profile": "phase_hampel_shift", "hidden_size": 96, "attention_dim": 32, "num_layers": 1, "dropout": 0.3, "lr": 5e-4, "noise_sigma": 0.01, "purpose": "denoise+noise regularization"},
    {"run": "S2_A09", "experiment": "S2_A09_raw_shift_bigru_h64_a32", "profile": "raw_shift", "hidden_size": 64, "attention_dim": 32, "num_layers": 1, "dropout": 0.2, "lr": 1e-3, "bidirectional": True, "purpose": "BiGRU small probe"},
    {"run": "S2_A10", "experiment": "S2_A10_raw_shift_h96_a48_lr0005", "profile": "raw_shift", "hidden_size": 96, "attention_dim": 48, "num_layers": 1, "dropout": 0.3, "lr": 5e-4, "purpose": "A12-style regularized"},
]

MULTI_PLANNED = [
    {"run": "S3_M01", "experiment": "S3_M01_top_attention_linear", "profile": "selected", "cell_head_type": "linear", "cell_loss_type": "cross_entropy", "purpose": "multi baseline từ top attention"},
    {"run": "S3_M02", "experiment": "S3_M02_top_attention_mlp", "profile": "selected", "cell_head_type": "mlp", "cell_loss_type": "cross_entropy", "purpose": "MLP cell head"},
    {"run": "S3_M03", "experiment": "S3_M03_raw_shift_h128_mlp", "profile": "raw_shift", "cell_head_type": "mlp", "cell_loss_type": "cross_entropy", "purpose": "raw_shift h128 MLP"},
    {"run": "S3_M04", "experiment": "S3_M04_raw_shift_h128_mlp_cell2", "profile": "raw_shift", "cell_head_type": "mlp", "cell_loss_type": "cross_entropy", "loss_weight_cell": 2.0, "purpose": "cell loss x2"},
    {"run": "S3_M05", "experiment": "S3_M05_raw_shift_h128_mlp_focal", "profile": "raw_shift", "cell_head_type": "mlp", "cell_loss_type": "focal", "cell_focal_gamma": 2.0, "loss_weight_cell": 2.0, "purpose": "focal cell hard examples"},
    {"run": "S3_M06", "experiment": "S3_M06_raw_shift_h128_mlp_effective", "profile": "raw_shift", "cell_head_type": "mlp", "cell_loss_type": "focal", "cell_class_weighting": "effective_number", "cell_focal_gamma": 1.5, "loss_weight_cell": 2.0, "purpose": "effective-number class balance"},
    {"run": "S3_M07", "experiment": "S3_M07_phase_hampel_shift_mlp", "profile": "phase_hampel_shift", "cell_head_type": "mlp", "cell_loss_type": "cross_entropy", "purpose": "denoise multi baseline"},
    {"run": "S3_M08", "experiment": "S3_M08_phase_hampel_shift_mlp_noise", "profile": "phase_hampel_shift", "cell_head_type": "mlp", "cell_loss_type": "cross_entropy", "noise_sigma": 0.01, "dropout": 0.3, "purpose": "denoise+noise multi"},
    {"run": "S3_M09", "experiment": "S3_M09_raw_shift_mlp_l2_d03", "profile": "raw_shift", "cell_head_type": "mlp", "cell_loss_type": "cross_entropy", "num_layers": 2, "dropout": 0.3, "purpose": "stacked multi"},
    {"run": "S3_M10", "experiment": "S3_M10_raw_shift_bigru_mlp", "profile": "raw_shift", "cell_head_type": "mlp", "cell_loss_type": "cross_entropy", "bidirectional": True, "hidden_size": 64, "purpose": "BiGRU multi probe"},
]

FINAL_ATTENTION_PLANNED = [
    {"run": "S4_F01", "experiment": "S4_F01_attention_top_seed43", "profile": "selected", "purpose": "top attention seed check"},
    {"run": "S4_F02", "experiment": "S4_F02_attention_second_seed43", "profile": "selected", "purpose": "second attention seed check"},
]

FINAL_MULTI_PLANNED = [
    {"run": "S4_F03", "experiment": "S4_F03_multi_top_seed43", "profile": "selected", "purpose": "top multi seed 43"},
    {"run": "S4_F04", "experiment": "S4_F04_multi_top_seed44", "profile": "selected", "purpose": "top multi seed 44"},
    {"run": "S4_F05", "experiment": "S4_F05_multi_second_seed43", "profile": "selected", "purpose": "second multi seed 43"},
    {"run": "S4_F06", "experiment": "S4_F06_multi_linear_ablation", "profile": "selected", "purpose": "linear-vs-MLP ablation"},
    {"run": "S4_F07", "experiment": "S4_F07_multi_ce_ablation", "profile": "selected", "purpose": "CE-vs-focal ablation"},
    {"run": "S4_F08", "experiment": "S4_F08_multi_layer_ablation", "profile": "selected", "purpose": "num_layers ablation"},
]

ALL_PLANNED = (
    [{"stage": "Stage 1 GRU", **item} for item in GRU_PLANNED]
    + [{"stage": "Stage 2 Attention", **item} for item in ATTENTION_PLANNED]
    + [{"stage": "Stage 3 Multi", **item} for item in MULTI_PLANNED]
    + [{"stage": "Stage 4 Attention Final", **item} for item in FINAL_ATTENTION_PLANNED]
    + [{"stage": "Stage 4 Multi Final", **item} for item in FINAL_MULTI_PLANNED]
)

experiment_matrix = pd.DataFrame(ALL_PLANNED)
display(experiment_matrix)
print(f"Total planned training runs: {len(ALL_PLANNED)}")


,stage,run,experiment,profile,hidden_size,num_layers,dropout,batch_size,lr,purpose,attention_dim,noise_sigma,bidirectional,cell_head_type,cell_loss_type,loss_weight_cell,cell_focal_gamma,cell_class_weighting
0,Stage 1 GRU,S1_G01,S1_G01_raw_h64,raw,64,1,0.0,32.0,0.001,raw baseline đối chứng,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Stage 1 GRU,S1_G02,S1_G02_raw_shift_h64_anchor,raw_shift,64,1,0.0,32.0,0.001,anchor ổn định từ G02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Stage 1 GRU,S1_G03,S1_G03_raw_shift_h96,raw_shift,96,1,0.0,32.0,0.001,capacity trung bình,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Stage 1 GRU,S1_G04,S1_G04_raw_shift_h128,raw_shift,128,1,0.0,32.0,0.001,capacity cao kiểm tra overfit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Stage 1 GRU,S1_G05,S1_G05_raw_shift_h64_l2_d03,raw_shift,64,2,0.3,32.0,0.001,stacked GRU nhỏ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Stage 1 GRU,S1_G06,S1_G06_raw_shift_h96_l2_d03,raw_shift,96,2,0.3,32.0,0.001,stacked GRU trung bình,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Stage 1 GRU,S1_G07,S1_G07_raw_shift_h96_b64_lr0005,raw_shift,96,1,0.0,64.0,0.0005,batch/lr ổn định,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Stage 2 Attention,S2_A01,S2_A01_top_gru_a32,selected,selected,selected,0.2,NaN,selected,top GRU -> attention a32,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Stage 2 Attention,S2_A02,S2_A02_top_gru_a48,selected,selected,selected,0.2,NaN,selected,top GRU -> attention a48,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Stage 2 Attention,S2_A03,S2_A03_second_gru_a32,selected,selected,selected,0.2,NaN,selected,second GRU branch,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Total planned training runs: 35


## Stage 1 - GRU baseline/capacity screening (`S1_G01`-`S1_G07`)

Mục tiêu: xác nhận anchor `raw_shift`, so sánh raw baseline, hidden size, batch/LR, và `num_layers=2`.

### S1_G01 - raw h64 baseline


In [3]:
config = train_gru.TrainConfig(experiment_name="S1_G01_raw_h64")
config.profile = "raw"
config.hidden_size = 64
config.num_layers = 1
config.dropout = 0.0
config.batch_size = 32
config.learning_rate = 0.001
config.bidirectional = False
timed_train("Stage 1 GRU", train_gru.train, config)


GRU baseline:  52%|█████▎    | 42/80 [00:10<00:09,  4.15it/s, val_f1=0.624, val_loss=0.934]


Stage 1 GRU | S1_G01_raw_h64 | 0.23 min


{'phase_name': 'GRU baseline',
 'profile': 'raw',
 'experiment_name': 'S1_G01_raw_h64',
 'hidden_size': 64,
 'dropout': 0.0,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 31,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 43,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\gru_baseline\\S1_G01_raw_h64',
 'best_val_accuracy': 0.7142857142857143,
 'best_val_macro_f1': 0.6500496671768794,
 'best_val_weighted_f1': 0.7151137727030707,
 'best_val_macro_precision': 0.6543400305858776,
 'best_val_macro_recall': 0.6522255178667212,
 'best_val_loss': 0.909488878931318,
 'test_accuracy': 0.6115107913669064,
 'test_macro_f1': 0.5636919673850591,
 'test_weighted_f1': 0.6215516956927217,
 'test_macro_precision': 0.5623720297344585,
 'test_macro_recall': 0.5820600358674775,
 'test_loss': 1.2032516645870621,
 'elapsed_minutes': 0.23240029666631018}

### S1_G02 - raw_shift h64 anchor


In [4]:
config = train_gru.TrainConfig(experiment_name="S1_G02_raw_shift_h64_anchor")
config.profile = "raw_shift"
config.hidden_size = 64
config.num_layers = 1
config.dropout = 0.0
config.batch_size = 32
config.learning_rate = 0.001
config.bidirectional = False
timed_train("Stage 1 GRU", train_gru.train, config)


GRU baseline:  44%|████▍     | 35/80 [00:33<00:43,  1.03it/s, val_f1=0.66, val_loss=1.61]  


Stage 1 GRU | S1_G02_raw_shift_h64_anchor | 0.59 min


{'phase_name': 'GRU baseline',
 'profile': 'raw_shift',
 'experiment_name': 'S1_G02_raw_shift_h64_anchor',
 'hidden_size': 64,
 'dropout': 0.0,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 24,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 36,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\gru_baseline\\S1_G02_raw_shift_h64_anchor',
 'best_val_accuracy': 0.7571428571428571,
 'best_val_macro_f1': 0.694430085958393,
 'best_val_weighted_f1': 0.7563742329608623,
 'best_val_macro_precision': 0.7162799189114979,
 'best_val_macro_recall': 0.6937977318587009,
 'best_val_loss': 1.2457377399717058,
 'test_accuracy': 0.7338129496402878,
 'test_macro_f1': 0.6876813841098146,
 'test_weighted_f1': 0.7349378754496386,
 'test_macro_precision': 0.699577565291851,
 'test_macro_recall': 0.6847184459427673,
 'test_loss': 1.4165153829313868,
 'elapsed_minutes': 0.5933822699997109}

### S1_G03 - raw_shift h96


In [5]:
config = train_gru.TrainConfig(experiment_name="S1_G03_raw_shift_h96")
config.profile = "raw_shift"
config.hidden_size = 96
config.num_layers = 1
config.dropout = 0.0
config.batch_size = 32
config.learning_rate = 0.001
config.bidirectional = False
timed_train("Stage 1 GRU", train_gru.train, config)


GRU baseline:  29%|██▉       | 23/80 [00:23<00:59,  1.04s/it, val_f1=0.633, val_loss=1.52]


Stage 1 GRU | S1_G03_raw_shift_h96 | 0.43 min


{'phase_name': 'GRU baseline',
 'profile': 'raw_shift',
 'experiment_name': 'S1_G03_raw_shift_h96',
 'hidden_size': 96,
 'dropout': 0.0,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 12,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 24,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\gru_baseline\\S1_G03_raw_shift_h96',
 'best_val_accuracy': 0.7214285714285714,
 'best_val_macro_f1': 0.6695361963513368,
 'best_val_weighted_f1': 0.7203889202278797,
 'best_val_macro_precision': 0.6595458582546101,
 'best_val_macro_recall': 0.6964924267579372,
 'best_val_loss': 1.007115295955113,
 'test_accuracy': 0.6762589928057554,
 'test_macro_f1': 0.6111958873590663,
 'test_weighted_f1': 0.6783597242431304,
 'test_macro_precision': 0.6096153081115487,
 'test_macro_recall': 0.6283276765951463,
 'test_loss': 1.2017238783321793,
 'elapsed_minutes': 0.42739277333312203}

### S1_G04 - raw_shift h128


In [6]:
config = train_gru.TrainConfig(experiment_name="S1_G04_raw_shift_h128")
config.profile = "raw_shift"
config.hidden_size = 128
config.num_layers = 1
config.dropout = 0.0
config.batch_size = 32
config.learning_rate = 0.001
config.bidirectional = False
timed_train("Stage 1 GRU", train_gru.train, config)


GRU baseline:  60%|██████    | 48/80 [00:47<00:31,  1.00it/s, val_f1=0.704, val_loss=1.93]


Stage 1 GRU | S1_G04_raw_shift_h128 | 0.82 min


{'phase_name': 'GRU baseline',
 'profile': 'raw_shift',
 'experiment_name': 'S1_G04_raw_shift_h128',
 'hidden_size': 128,
 'dropout': 0.0,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 37,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 49,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\gru_baseline\\S1_G04_raw_shift_h128',
 'best_val_accuracy': 0.8071428571428572,
 'best_val_macro_f1': 0.7474483086826386,
 'best_val_weighted_f1': 0.8018652674872541,
 'best_val_macro_precision': 0.7598928370480095,
 'best_val_macro_recall': 0.7442929299379326,
 'best_val_loss': 1.8148978224822454,
 'test_accuracy': 0.7194244604316546,
 'test_macro_f1': 0.6551289817639836,
 'test_weighted_f1': 0.7179465900159934,
 'test_macro_precision': 0.6681166219611283,
 'test_macro_recall': 0.6573911874959644,
 'test_loss': 2.284925652922486,
 'elapsed_minutes': 0.8241323250006342}

### S1_G05 - raw_shift h64 num_layers2 dropout03


In [7]:
config = train_gru.TrainConfig(experiment_name="S1_G05_raw_shift_h64_l2_d03")
config.profile = "raw_shift"
config.hidden_size = 64
config.num_layers = 2
config.dropout = 0.3
config.batch_size = 32
config.learning_rate = 0.001
config.bidirectional = False
timed_train("Stage 1 GRU", train_gru.train, config)


GRU baseline:  57%|█████▊    | 46/80 [00:42<00:31,  1.09it/s, val_f1=0.703, val_loss=2.04]


Stage 1 GRU | S1_G05_raw_shift_h64_l2_d03 | 0.73 min


{'phase_name': 'GRU baseline',
 'profile': 'raw_shift',
 'experiment_name': 'S1_G05_raw_shift_h64_l2_d03',
 'hidden_size': 64,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'num_layers': 2,
 'bidirectional': False,
 'best_epoch': 35,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 47,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\gru_baseline\\S1_G05_raw_shift_h64_l2_d03',
 'best_val_accuracy': 0.7714285714285715,
 'best_val_macro_f1': 0.7217172499498333,
 'best_val_weighted_f1': 0.7638925306192857,
 'best_val_macro_precision': 0.7700175764085538,
 'best_val_macro_recall': 0.7061768856478762,
 'best_val_loss': 1.8065530572618758,
 'test_accuracy': 0.7050359712230215,
 'test_macro_f1': 0.6593515033589548,
 'test_weighted_f1': 0.7028508039895499,
 'test_macro_precision': 0.6837650444793303,
 'test_macro_recall': 0.6526767185458713,
 'test_loss': 2.36928478505114,
 'elapsed_minutes': 0.730706961666389}

### S1_G06 - raw_shift h96 num_layers2 dropout03


In [8]:
config = train_gru.TrainConfig(experiment_name="S1_G06_raw_shift_h96_l2_d03")
config.profile = "raw_shift"
config.hidden_size = 96
config.num_layers = 2
config.dropout = 0.3
config.batch_size = 32
config.learning_rate = 0.001
config.bidirectional = False
timed_train("Stage 1 GRU", train_gru.train, config)


GRU baseline:  35%|███▌      | 28/80 [00:27<00:51,  1.00it/s, val_f1=0.725, val_loss=1.61] 
                                

Stage 1 GRU | S1_G06_raw_shift_h96_l2_d03 | 0.49 min


{'phase_name': 'GRU baseline',
 'profile': 'raw_shift',
 'experiment_name': 'S1_G06_raw_shift_h96_l2_d03',
 'hidden_size': 96,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'num_layers': 2,
 'bidirectional': False,
 'best_epoch': 17,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 29,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\gru_baseline\\S1_G06_raw_shift_h96_l2_d03',
 'best_val_accuracy': 0.8071428571428572,
 'best_val_macro_f1': 0.7435100846865552,
 'best_val_weighted_f1': 0.8001213165919049,
 'best_val_macro_precision': 0.7585386835386835,
 'best_val_macro_recall': 0.7365273236954356,
 'best_val_loss': 1.3434253249849593,
 'test_accuracy': 0.7194244604316546,
 'test_macro_f1': 0.6592607010498235,
 'test_weighted_f1': 0.7172792909792959,
 'test_macro_precision': 0.6928023797096746,
 'test_macro_recall': 0.6661077621971937,
 'test_loss': 1.577546914704412,
 'elapsed_minutes': 0.49158248833361234}

### S1_G07 - raw_shift h96 batch64 lr0005


In [9]:
config = train_gru.TrainConfig(experiment_name="S1_G07_raw_shift_h96_b64_lr0005")
config.profile = "raw_shift"
config.hidden_size = 96
config.num_layers = 1
config.dropout = 0.0
config.batch_size = 64
config.learning_rate = 0.0005
config.bidirectional = False
timed_train("Stage 1 GRU", train_gru.train, config)


GRU baseline:  38%|███▊      | 30/80 [00:22<00:38,  1.31it/s, val_f1=0.65, val_loss=1.12]  


Stage 1 GRU | S1_G07_raw_shift_h96_b64_lr0005 | 0.41 min


{'phase_name': 'GRU baseline',
 'profile': 'raw_shift',
 'experiment_name': 'S1_G07_raw_shift_h96_b64_lr0005',
 'hidden_size': 96,
 'dropout': 0.0,
 'batch_size': 64,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 19,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 31,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\gru_baseline\\S1_G07_raw_shift_h96_b64_lr0005',
 'best_val_accuracy': 0.7357142857142858,
 'best_val_macro_f1': 0.7007770449610343,
 'best_val_weighted_f1': 0.7393693563795288,
 'best_val_macro_precision': 0.6871477553626474,
 'best_val_macro_recall': 0.7336314845086971,
 'best_val_loss': 0.914740766797747,
 'test_accuracy': 0.6330935251798561,
 'test_macro_f1': 0.581302056438717,
 'test_weighted_f1': 0.6461628153190414,
 'test_macro_precision': 0.5914393125671321,
 'test_macro_recall': 0.6035177447107155,
 'test_loss': 1.2286402480207759,
 'elapsed_minutes': 0.4068694500000371}

## Stage 1 selection

Chọn top 3 GRU theo `best_val_macro_f1`, tie-break bằng `best_val_loss`.

In [10]:
gru_status = load_planned_status("Stage 1 GRU", GRU_PLANNED, GRU_RUNS_ROOT, ["best_val_macro_f1", "best_val_loss", "test_macro_f1", "test_accuracy", "elapsed_minutes", "epochs_ran"])
display(gru_status)
top_gru_configs, top_gru_ranking = select_top_configs(GRU_PLANNED, GRU_RUNS_ROOT, "best_val_macro_f1", "best_val_loss", 3)
display(top_gru_ranking)


,stage,run,experiment,profile,hidden_size,num_layers,dropout,batch_size,lr,purpose,status,run_dir,best_val_macro_f1,best_val_loss,test_macro_f1,test_accuracy,elapsed_minutes,epochs_ran
0,Stage 1 GRU,S1_G01,S1_G01_raw_h64,raw,64,1,0.0,32,0.0010,raw baseline đối chứng,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.650050,0.909489,0.563692,0.611511,0.232400,43
1,Stage 1 GRU,S1_G02,S1_G02_raw_shift_h64_anchor,raw_shift,64,1,0.0,32,0.0010,anchor ổn định từ G02,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.694430,1.245738,0.687681,0.733813,0.593382,36
2,Stage 1 GRU,S1_G03,S1_G03_raw_shift_h96,raw_shift,96,1,0.0,32,0.0010,capacity trung bình,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.669536,1.007115,0.611196,0.676259,0.427393,24
3,Stage 1 GRU,S1_G04,S1_G04_raw_shift_h128,raw_shift,128,1,0.0,32,0.0010,capacity cao kiểm tra overfit,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.747448,1.814898,0.655129,0.719424,0.824132,49
4,Stage 1 GRU,S1_G05,S1_G05_raw_shift_h64_l2_d03,raw_shift,64,2,0.3,32,0.0010,stacked GRU nhỏ,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.721717,1.806553,0.659352,0.705036,0.730707,47
5,Stage 1 GRU,S1_G06,S1_G06_raw_shift_h96_l2_d03,raw_shift,96,2,0.3,32,0.0010,stacked GRU trung bình,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.743510,1.343425,0.659261,0.719424,0.491582,29
6,Stage 1 GRU,S1_G07,S1_G07_raw_shift_h96_b64_lr0005,raw_shift,96,1,0.0,64,0.0005,batch/lr ổn định,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.700777,0.914741,0.581302,0.633094,0.406869,31


,stage,run,experiment,profile,hidden_size,num_layers,dropout,batch_size,lr,purpose,status,run_dir,best_val_macro_f1,best_val_loss
3,selection,S1_G04,S1_G04_raw_shift_h128,raw_shift,128,1,0.0,32,0.001,capacity cao kiểm tra overfit,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.747448,1.814898
5,selection,S1_G06,S1_G06_raw_shift_h96_l2_d03,raw_shift,96,2,0.3,32,0.001,stacked GRU trung bình,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.743510,1.343425
4,selection,S1_G05,S1_G05_raw_shift_h64_l2_d03,raw_shift,64,2,0.3,32,0.001,stacked GRU nhỏ,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.721717,1.806553


## Stage 2 - Attention-GRU narrowing (`S2_A01`-`S2_A10`)

Mục tiêu: dùng top GRU làm backbone bank, sau đó thử attention dim, dropout, denoise branch, noise nhẹ và BiGRU nhỏ.

### S2_A01 - top GRU attention a32


In [11]:
selected = pick_config(top_gru_configs, 0, fallback_config(profile='raw_shift', hidden_size=96, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.0, bidirectional=False))
config = train_attention.TrainConfig(experiment_name="S2_A01_top_gru_a32")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 32
config.dropout = 0.2


config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  33%|███▎      | 33/100 [00:33<01:07,  1.00s/it, val_f1=0.699, val_loss=1.79]


Stage 2 Attention | S2_A01_top_gru_a32 | 0.58 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A01_top_gru_a32',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.2,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 19,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 34,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A01_top_gru_a32',
 'best_val_accuracy': 0.7714285714285715,
 'best_val_macro_f1': 0.724154080288172,
 'best_val_weighted_f1': 0.7734564948879641,
 'best_val_macro_precision': 0.7418883382699172,
 'best_val_macro_recall': 0.7213297553063779,
 'best_val_loss': 1.542205079964229,
 'test_accuracy': 0.6618705035971223,
 'test_macro_f1': 0.5874128987733476,
 'test_weighted_f1': 0.6537015548851242,
 'test_macro_precision': 0.6119960984717546,
 'test_macro_recall': 0.5930423631314647,
 'test_loss': 2.0532512767709417,
 'elapsed_minutes': 0.5775039

### S2_A02 - top GRU attention a48


In [12]:
selected = pick_config(top_gru_configs, 0, fallback_config(profile='raw_shift', hidden_size=96, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.0, bidirectional=False))
config = train_attention.TrainConfig(experiment_name="S2_A02_top_gru_a48")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 48
config.dropout = 0.2


config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  36%|███▌      | 36/100 [00:35<01:03,  1.01it/s, val_f1=0.647, val_loss=2.07]


Stage 2 Attention | S2_A02_top_gru_a48 | 0.62 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A02_top_gru_a48',
 'hidden_size': 128,
 'attention_dim': 48,
 'dropout': 0.2,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 22,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 37,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A02_top_gru_a48',
 'best_val_accuracy': 0.7571428571428571,
 'best_val_macro_f1': 0.6919557363388052,
 'best_val_weighted_f1': 0.7540689308943048,
 'best_val_macro_precision': 0.699389870890885,
 'best_val_macro_recall': 0.6970232241832903,
 'best_val_loss': 1.4964902222156524,
 'test_accuracy': 0.6762589928057554,
 'test_macro_f1': 0.6246969196217317,
 'test_weighted_f1': 0.6769996852825327,
 'test_macro_precision': 0.6247957424728133,
 'test_macro_recall': 0.6292590111169437,
 'test_loss': 2.009026150051638,
 'elapsed_minutes': 0.6228333

### S2_A03 - second GRU attention a32


In [13]:
selected = pick_config(top_gru_configs, 1, fallback_config(profile='raw_shift', hidden_size=96, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.0, bidirectional=False))
config = train_attention.TrainConfig(experiment_name="S2_A03_second_gru_a32")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 32
config.dropout = 0.2


config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  35%|███▌      | 35/100 [00:34<01:03,  1.02it/s, val_f1=0.698, val_loss=1.95] 


Stage 2 Attention | S2_A03_second_gru_a32 | 0.60 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A03_second_gru_a32',
 'hidden_size': 96,
 'attention_dim': 32,
 'dropout': 0.2,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'best_epoch': 21,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 36,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A03_second_gru_a32',
 'best_val_accuracy': 0.7642857142857142,
 'best_val_macro_f1': 0.7087609052945189,
 'best_val_weighted_f1': 0.758510546409706,
 'best_val_macro_precision': 0.7261613386613386,
 'best_val_macro_recall': 0.701487509897576,
 'best_val_loss': 1.5349908896854945,
 'test_accuracy': 0.7050359712230215,
 'test_macro_f1': 0.6403426614590116,
 'test_weighted_f1': 0.7069701821548515,
 'test_macro_precision': 0.6348072562358277,
 'test_macro_recall': 0.6480369441263757,
 'test_loss': 1.939577421696066,
 'elapsed_minutes': 0.598

### S2_A04 - second GRU stacked attention


In [14]:
selected = pick_config(top_gru_configs, 1, fallback_config(profile='raw_shift', hidden_size=96, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.0, bidirectional=False))
config = train_attention.TrainConfig(experiment_name="S2_A04_second_gru_l2_d03")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 32
config.dropout = 0.3
config.num_layers = 2

config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  31%|███       | 31/100 [00:30<01:08,  1.01it/s, val_f1=0.708, val_loss=1.68]


Stage 2 Attention | S2_A04_second_gru_l2_d03 | 0.54 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A04_second_gru_l2_d03',
 'hidden_size': 96,
 'attention_dim': 32,
 'dropout': 0.3,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'best_epoch': 17,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 32,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A04_second_gru_l2_d03',
 'best_val_accuracy': 0.7642857142857142,
 'best_val_macro_f1': 0.7187937195080052,
 'best_val_weighted_f1': 0.7673210334638907,
 'best_val_macro_precision': 0.7281403345082591,
 'best_val_macro_recall': 0.7172821256162039,
 'best_val_loss': 1.310521913426263,
 'test_accuracy': 0.6906474820143885,
 'test_macro_f1': 0.6415332290735084,
 'test_weighted_f1': 0.688188474782552,
 'test_macro_precision': 0.6845571561958117,
 'test_macro_recall': 0.6267911199435001,
 'test_loss': 1.658483633034521,
 'elapsed_minutes':

### S2_A05 - raw_shift h128 a32 dropout03


In [15]:
selected = fallback_config(profile='raw_shift', hidden_size=128, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.3, bidirectional=False)
config = train_attention.TrainConfig(experiment_name="S2_A05_raw_shift_h128_a32_d03")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 32
config.dropout = 0.3


config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  34%|███▍      | 34/100 [00:34<01:07,  1.02s/it, val_f1=0.668, val_loss=2.03] 


Stage 2 Attention | S2_A05_raw_shift_h128_a32_d03 | 0.61 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A05_raw_shift_h128_a32_d03',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 20,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 35,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A05_raw_shift_h128_a32_d03',
 'best_val_accuracy': 0.7571428571428571,
 'best_val_macro_f1': 0.7100855120822464,
 'best_val_weighted_f1': 0.7568829028347247,
 'best_val_macro_precision': 0.7530416077774004,
 'best_val_macro_recall': 0.6866331639038593,
 'best_val_loss': 1.4498113802501134,
 'test_accuracy': 0.6330935251798561,
 'test_macro_f1': 0.5744179290097657,
 'test_weighted_f1': 0.6399807797546748,
 'test_macro_precision': 0.5971778221778222,
 'test_macro_recall': 0.5675954349805516,
 'test_loss': 2.177431287525369,
 'elap

### S2_A06 - raw_shift h96 a64 dropout03


In [16]:
selected = fallback_config(profile='raw_shift', hidden_size=96, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.3, bidirectional=False)
config = train_attention.TrainConfig(experiment_name="S2_A06_raw_shift_h96_a64_d03")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 64
config.dropout = 0.3


config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  39%|███▉      | 39/100 [00:38<00:59,  1.02it/s, val_f1=0.625, val_loss=2.41]
                                

Stage 2 Attention | S2_A06_raw_shift_h96_a64_d03 | 0.66 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A06_raw_shift_h96_a64_d03',
 'hidden_size': 96,
 'attention_dim': 64,
 'dropout': 0.3,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 25,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 40,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A06_raw_shift_h96_a64_d03',
 'best_val_accuracy': 0.7714285714285715,
 'best_val_macro_f1': 0.7031875267974221,
 'best_val_weighted_f1': 0.774120276948618,
 'best_val_macro_precision': 0.7503170771027914,
 'best_val_macro_recall': 0.6863713570534598,
 'best_val_loss': 1.5271647231919425,
 'test_accuracy': 0.6402877697841727,
 'test_macro_f1': 0.5733890936906729,
 'test_weighted_f1': 0.643306722488417,
 'test_macro_precision': 0.5809523809523809,
 'test_macro_recall': 0.576637822388998,
 'test_loss': 2.4095903352010164,
 'elapsed_m

### S2_A07 - phase_hampel_shift h96 a32


In [17]:
selected = fallback_config(profile='phase_hampel_shift', hidden_size=96, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.2, bidirectional=False)
config = train_attention.TrainConfig(experiment_name="S2_A07_phase_hampel_shift_h96_a32")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 32
config.dropout = 0.2


config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  30%|███       | 30/100 [00:30<01:10,  1.01s/it, val_f1=0.607, val_loss=2.2] 
                                

Stage 2 Attention | S2_A07_phase_hampel_shift_h96_a32 | 0.53 min


{'phase_name': 'Attention-GRU',
 'profile': 'phase_hampel_shift',
 'experiment_name': 'S2_A07_phase_hampel_shift_h96_a32',
 'hidden_size': 96,
 'attention_dim': 32,
 'dropout': 0.2,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 16,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 31,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A07_phase_hampel_shift_h96_a32',
 'best_val_accuracy': 0.7357142857142858,
 'best_val_macro_f1': 0.6589033715190197,
 'best_val_weighted_f1': 0.7291288054226375,
 'best_val_macro_precision': 0.7011505919891013,
 'best_val_macro_recall': 0.6475034162601211,
 'best_val_loss': 1.5809775467429843,
 'test_accuracy': 0.6906474820143885,
 'test_macro_f1': 0.6494030417339871,
 'test_weighted_f1': 0.6943804982215199,
 'test_macro_precision': 0.6772394272394272,
 'test_macro_recall': 0.6714864785774777,
 'test_loss': 1.6270752

### S2_A08 - phase_hampel_shift noise lr0005


In [18]:
selected = fallback_config(profile='phase_hampel_shift', hidden_size=96, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, bidirectional=False)
config = train_attention.TrainConfig(experiment_name="S2_A08_phase_hampel_shift_noise_lr0005")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 32
config.dropout = 0.3

config.learning_rate = 0.0005
config.noise_sigma = 0.01
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  34%|███▍      | 34/100 [02:04<04:02,  3.67s/it, val_f1=0.602, val_loss=2.12]


Stage 2 Attention | S2_A08_phase_hampel_shift_noise_lr0005 | 2.11 min


{'phase_name': 'Attention-GRU',
 'profile': 'phase_hampel_shift',
 'experiment_name': 'S2_A08_phase_hampel_shift_noise_lr0005',
 'hidden_size': 96,
 'attention_dim': 32,
 'dropout': 0.3,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.01,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 20,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 35,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A08_phase_hampel_shift_noise_lr0005',
 'best_val_accuracy': 0.7285714285714285,
 'best_val_macro_f1': 0.6505605416769883,
 'best_val_weighted_f1': 0.7277201674320521,
 'best_val_macro_precision': 0.6623629208768528,
 'best_val_macro_recall': 0.6448398508339507,
 'best_val_loss': 1.3022922890526907,
 'test_accuracy': 0.6187050359712231,
 'test_macro_f1': 0.534851041057422,
 'test_weighted_f1': 0.6270765927221023,
 'test_macro_precision': 0.5431901431901431,
 'test_macro_recall': 0.5446034813393117,
 'test_loss'

### S2_A09 - raw_shift BiGRU h64 a32


In [19]:
selected = fallback_config(profile='raw_shift', hidden_size=64, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.2, bidirectional=True)
config = train_attention.TrainConfig(experiment_name="S2_A09_raw_shift_bigru_h64_a32")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 32
config.dropout = 0.2


config.noise_sigma = 0.0
config.bidirectional = True or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  20%|██        | 20/100 [00:28<01:53,  1.42s/it, val_f1=0.684, val_loss=1.24] 


Stage 2 Attention | S2_A09_raw_shift_bigru_h64_a32 | 0.50 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A09_raw_shift_bigru_h64_a32',
 'hidden_size': 64,
 'attention_dim': 32,
 'dropout': 0.2,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': True,
 'best_epoch': 6,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 21,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A09_raw_shift_bigru_h64_a32',
 'best_val_accuracy': 0.7357142857142858,
 'best_val_macro_f1': 0.7035038882259392,
 'best_val_weighted_f1': 0.7365086145075052,
 'best_val_macro_precision': 0.6928639192023948,
 'best_val_macro_recall': 0.7305568503997344,
 'best_val_loss': 0.8724282435008458,
 'test_accuracy': 0.6115107913669064,
 'test_macro_f1': 0.5770289184562885,
 'test_weighted_f1': 0.6266428527030811,
 'test_macro_precision': 0.5793650793650793,
 'test_macro_recall': 0.6163156653318357,
 'test_loss': 1.031600962439887,
 'elaps

### S2_A10 - raw_shift h96 a48 lr0005


In [20]:
selected = fallback_config(profile='raw_shift', hidden_size=96, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, bidirectional=False)
config = train_attention.TrainConfig(experiment_name="S2_A10_raw_shift_h96_a48_lr0005")
apply_common(config, selected, ["profile", "hidden_size", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "bidirectional"])
config.attention_dim = 48
config.dropout = 0.3

config.learning_rate = 0.0005
config.noise_sigma = 0.0
config.bidirectional = False or config.bidirectional
timed_train("Stage 2 Attention", train_attention.train, config)


Attention-GRU:  39%|███▉      | 39/100 [00:52<01:21,  1.34s/it, val_f1=0.661, val_loss=1.63] 


Stage 2 Attention | S2_A10_raw_shift_h96_a48_lr0005 | 0.90 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S2_A10_raw_shift_h96_a48_lr0005',
 'hidden_size': 96,
 'attention_dim': 48,
 'dropout': 0.3,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 25,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 40,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S2_A10_raw_shift_h96_a48_lr0005',
 'best_val_accuracy': 0.7428571428571429,
 'best_val_macro_f1': 0.7103023846587375,
 'best_val_weighted_f1': 0.7464292776161909,
 'best_val_macro_precision': 0.7426152463888313,
 'best_val_macro_recall': 0.7130796467523179,
 'best_val_loss': 1.1018740854093008,
 'test_accuracy': 0.7050359712230215,
 'test_macro_f1': 0.6741193306658203,
 'test_weighted_f1': 0.7055210791713052,
 'test_macro_precision': 0.6963737659681961,
 'test_macro_recall': 0.6767603304852707,
 'test_loss': 1.5312501826732279,
 

## Stage 2 selection

Chọn top 4 Attention-GRU theo validation macro-F1 để truyền sang multi-output.

In [21]:
attention_status = load_planned_status("Stage 2 Attention", ATTENTION_PLANNED, ATTENTION_RUNS_ROOT, ["best_val_macro_f1", "best_val_loss", "test_macro_f1", "test_accuracy", "elapsed_minutes", "epochs_ran"])
display(attention_status)
top_attention_configs, top_attention_ranking = select_top_configs(ATTENTION_PLANNED, ATTENTION_RUNS_ROOT, "best_val_macro_f1", "best_val_loss", 4)
display(top_attention_ranking)


,stage,run,experiment,profile,hidden_size,attention_dim,num_layers,dropout,lr,purpose,status,run_dir,best_val_macro_f1,best_val_loss,test_macro_f1,test_accuracy,elapsed_minutes,epochs_ran,noise_sigma,bidirectional
0,Stage 2 Attention,S2_A01,S2_A01_top_gru_a32,selected,selected,32,selected,0.2,selected,top GRU -> attention a32,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.724154,1.542205,0.587413,0.661871,0.577504,34,NaN,NaN
1,Stage 2 Attention,S2_A02,S2_A02_top_gru_a48,selected,selected,48,selected,0.2,selected,top GRU -> attention a48,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.691956,1.496490,0.624697,0.676259,0.622833,37,NaN,NaN
2,Stage 2 Attention,S2_A03,S2_A03_second_gru_a32,selected,selected,32,selected,0.2,selected,second GRU branch,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.708761,1.534991,0.640343,0.705036,0.598936,36,NaN,NaN
3,Stage 2 Attention,S2_A04,S2_A04_second_gru_l2_d03,selected,selected,32,2,0.3,selected,stacked attention branch,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.718794,1.310522,0.641533,0.690647,0.537515,32,NaN,NaN
4,Stage 2 Attention,S2_A05,S2_A05_raw_shift_h128_a32_d03,raw_shift,128,32,1,0.3,0.001,h128 regularized,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.710086,1.449811,0.574418,0.633094,0.607337,35,NaN,NaN
5,Stage 2 Attention,S2_A06,S2_A06_raw_shift_h96_a64_d03,raw_shift,96,64,1,0.3,0.001,attention width upper,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.703188,1.527165,0.573389,0.640288,0.663559,40,NaN,NaN
6,Stage 2 Attention,S2_A07,S2_A07_phase_hampel_shift_h96_a32,phase_hampel_shift,96,32,1,0.2,0.001,denoise+shift stability,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.658903,1.580978,0.649403,0.690647,0.531154,31,NaN,NaN
7,Stage 2 Attention,S2_A08,S2_A08_phase_hampel_shift_noise_lr0005,phase_hampel_shift,96,32,1,0.3,0.0005,denoise+noise regularization,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.650561,1.302292,0.534851,0.618705,2.106173,35,0.01,NaN
8,Stage 2 Attention,S2_A09,S2_A09_raw_shift_bigru_h64_a32,raw_shift,64,32,1,0.2,0.001,BiGRU small probe,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.703504,0.872428,0.577029,0.611511,0.502108,21,NaN,True
9,Stage 2 Attention,S2_A10,S2_A10_raw_shift_h96_a48_lr0005,raw_shift,96,48,1,0.3,0.0005,A12-style regularized,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.710302,1.101874,0.674119,0.705036,0.902489,40,NaN,NaN


,stage,run,experiment,profile,hidden_size,attention_dim,num_layers,dropout,lr,purpose,status,run_dir,best_val_macro_f1,best_val_loss,noise_sigma,bidirectional
0,selection,S2_A01,S2_A01_top_gru_a32,selected,selected,32,selected,0.2,selected,top GRU -> attention a32,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.724154,1.542205,NaN,NaN
3,selection,S2_A04,S2_A04_second_gru_l2_d03,selected,selected,32,2,0.3,selected,stacked attention branch,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.718794,1.310522,NaN,NaN
9,selection,S2_A10,S2_A10_raw_shift_h96_a48_lr0005,raw_shift,96,48,1,0.3,0.0005,A12-style regularized,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.710302,1.101874,NaN,NaN
4,selection,S2_A05,S2_A05_raw_shift_h128_a32_d03,raw_shift,128,32,1,0.3,0.001,h128 regularized,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.710086,1.449811,NaN,NaN


## Stage 3 - Multi-output cell bottleneck attack (`S3_M01`-`S3_M10`)

Mục tiêu: dùng top Attention config, nâng cell head, thử cell weighting, focal/effective-number, denoise branch, stacked và BiGRU nhỏ.

### S3_M01 - top attention linear baseline


In [22]:
selected = pick_config(top_attention_configs, 0, fallback_config(profile='raw_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False))
config = train_multi.TrainConfig(experiment_name="S3_M01_top_attention_linear")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "linear"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 1.0

timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  23%|██▎       | 23/100 [00:33<01:53,  1.47s/it, loss=6.04, score=0.511]


Stage 3 Multi | S3_M01_top_attention_linear | 0.59 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M01_top_attention_linear',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.2,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'linear',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.0,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 9,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5210660711962192,
 'epochs_ran': 24,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M01_top_attention_linear',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.9,
 'val_pre

### S3_M02 - top attention MLP cell head


In [23]:
selected = pick_config(top_attention_configs, 0, fallback_config(profile='raw_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False))
config = train_multi.TrainConfig(experiment_name="S3_M02_top_attention_mlp")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 1.0

timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  61%|██████    | 61/100 [01:30<00:57,  1.48s/it, loss=8.63, score=0.544]
                                

Stage 3 Multi | S3_M02_top_attention_mlp | 1.54 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M02_top_attention_mlp',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.2,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.2,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 47,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5595243255883979,
 'epochs_ran': 62,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M02_top_attention_mlp',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.9285714285714286,
 '

### S3_M03 - raw_shift h128 MLP


In [24]:
selected = fallback_config(profile='raw_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M03_raw_shift_h128_mlp")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 1.0

timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  33%|███▎      | 33/100 [00:49<01:39,  1.49s/it, loss=5.89, score=0.489]


Stage 3 Multi | S3_M03_raw_shift_h128_mlp | 0.85 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M03_raw_shift_h128_mlp',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.3,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 19,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5176766631975883,
 'epochs_ran': 34,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M03_raw_shift_h128_mlp',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.8642857142857143,

### S3_M04 - raw_shift h128 MLP cell2


In [25]:
selected = fallback_config(profile='raw_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M04_raw_shift_h128_mlp_cell2")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 2.0

timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  53%|█████▎    | 53/100 [01:20<01:11,  1.52s/it, loss=13.8, score=0.527]


Stage 3 Multi | S3_M04_raw_shift_h128_mlp_cell2 | 1.37 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M04_raw_shift_h128_mlp_cell2',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.3,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 39,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5428810855391624,
 'epochs_ran': 54,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M04_raw_shift_h128_mlp_cell2',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.89285

### S3_M05 - raw_shift h128 MLP focal


In [26]:
selected = fallback_config(profile='raw_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M05_raw_shift_h128_mlp_focal")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "focal"
config.cell_focal_gamma = 2.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 2.0

timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  63%|██████▎   | 63/100 [01:39<00:58,  1.58s/it, loss=14.6, score=0.519]


Stage 3 Multi | S3_M05_raw_shift_h128_mlp_focal | 1.69 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M05_raw_shift_h128_mlp_focal',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.3,
 'cell_loss_type': 'focal',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 2.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 49,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5502262365006576,
 'epochs_ran': 64,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M05_raw_shift_h128_mlp_focal',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.8928571428571

### S3_M06 - raw_shift h128 MLP effective focal


In [27]:
selected = fallback_config(profile='raw_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M06_raw_shift_h128_mlp_effective")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "focal"
config.cell_focal_gamma = 1.5
config.cell_class_weighting = "effective_number"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 2.0

timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  30%|███       | 30/100 [00:46<01:49,  1.56s/it, loss=5.66, score=0.493]


Stage 3 Multi | S3_M06_raw_shift_h128_mlp_effective | 0.81 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M06_raw_shift_h128_mlp_effective',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.3,
 'cell_loss_type': 'focal',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 1.5,
 'cell_class_weighting': 'effective_number',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 16,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5059140109004859,
 'epochs_ran': 31,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M06_raw_shift_h128_mlp_effective',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.85,
 '

### S3_M07 - phase_hampel_shift MLP


In [28]:
selected = fallback_config(profile='phase_hampel_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.2, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M07_phase_hampel_shift_mlp")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 1.0

timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  79%|███████▉  | 79/100 [01:57<00:31,  1.49s/it, loss=12, score=0.526]  


Stage 3 Multi | S3_M07_phase_hampel_shift_mlp | 1.99 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M07_phase_hampel_shift_mlp',
 'profile': 'phase_hampel_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.2,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.2,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 65,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5375978722691467,
 'epochs_ran': 80,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M07_phase_hampel_shift_mlp',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.9

### S3_M08 - phase_hampel_shift MLP noise


In [29]:
selected = fallback_config(profile='phase_hampel_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.001, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M08_phase_hampel_shift_mlp_noise")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 1.0
config.noise_sigma = 0.01
config.dropout = 0.3
timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  41%|████      | 41/100 [02:56<04:13,  4.30s/it, loss=10.2, score=0.495]


Stage 3 Multi | S3_M08_phase_hampel_shift_mlp_noise | 2.97 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M08_phase_hampel_shift_mlp_noise',
 'profile': 'phase_hampel_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'noise_sigma': 0.01,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.3,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 27,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5105419754667277,
 'epochs_ran': 42,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M08_phase_hampel_shift_mlp_noise',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_a

### S3_M09 - raw_shift MLP num_layers2


In [30]:
selected = fallback_config(profile='raw_shift', hidden_size=128, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M09_raw_shift_mlp_l2_d03")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 1.0
config.num_layers = 2
config.dropout = 0.3
timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  73%|███████▎  | 73/100 [01:56<00:43,  1.60s/it, loss=9.36, score=0.543]
                                

Stage 3 Multi | S3_M09_raw_shift_mlp_l2_d03 | 1.97 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M09_raw_shift_mlp_l2_d03',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.3,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 59,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5641513552422177,
 'epochs_ran': 74,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M09_raw_shift_mlp_l2_d03',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.9071428571428

### S3_M10 - raw_shift BiGRU MLP


In [31]:
selected = fallback_config(profile='raw_shift', hidden_size=64, attention_dim=32, batch_size=32, learning_rate=0.0005, weight_decay=0.0001, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False)
config = train_multi.TrainConfig(experiment_name="S3_M10_raw_shift_bigru_mlp")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.cell_head_type = "mlp"
config.cell_head_dropout = max(config.dropout, 0.2) if config.cell_head_type == "mlp" else 0.0
config.cell_loss_type = "cross_entropy"
config.cell_focal_gamma = 0.0
config.cell_class_weighting = "inverse_frequency"
config.cell_label_smoothing = 0.0
config.loss_weight_cell = 1.0
config.bidirectional = True
timed_train("Stage 3 Multi" if config.experiment_name.startswith("S3_M") else "Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  43%|████▎     | 43/100 [01:12<01:35,  1.68s/it, loss=5.71, score=0.491]


Stage 3 Multi | S3_M10_raw_shift_bigru_mlp | 1.23 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S3_M10_raw_shift_bigru_mlp',
 'profile': 'raw_shift',
 'hidden_size': 64,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': True,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.3,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 29,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.49922259569429844,
 'epochs_ran': 44,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S3_M10_raw_shift_bigru_mlp',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.8642857142857143

## Stage 3 selection

Chọn top 3 Multi-output theo composite validation. Cell macro-F1 được hiển thị riêng vì đây là bottleneck chính.

In [32]:
multi_status = load_planned_status("Stage 3 Multi", MULTI_PLANNED, MULTI_RUNS_ROOT, ["best_val_composite_score", "val_loss", "val_cell_masked_macro_f1", "test_cell_masked_macro_f1", "test_pose_macro_f1", "elapsed_minutes", "epochs_ran"])
display(multi_status)
top_multi_configs, top_multi_ranking = select_top_configs(MULTI_PLANNED, MULTI_RUNS_ROOT, "best_val_composite_score", "val_loss", 3)
display(top_multi_ranking)


,stage,run,experiment,profile,cell_head_type,cell_loss_type,purpose,status,run_dir,best_val_composite_score,...,elapsed_minutes,epochs_ran,loss_weight_cell,cell_focal_gamma,cell_class_weighting,noise_sigma,dropout,num_layers,bidirectional,hidden_size
0,Stage 3 Multi,S3_M01,S3_M01_top_attention_linear,selected,linear,cross_entropy,multi baseline từ top attention,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.521066,...,0.594167,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Stage 3 Multi,S3_M02,S3_M02_top_attention_mlp,selected,mlp,cross_entropy,MLP cell head,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.559524,...,1.536474,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Stage 3 Multi,S3_M03,S3_M03_raw_shift_h128_mlp,raw_shift,mlp,cross_entropy,raw_shift h128 MLP,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.517677,...,0.846045,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Stage 3 Multi,S3_M04,S3_M04_raw_shift_h128_mlp_cell2,raw_shift,mlp,cross_entropy,cell loss x2,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.542881,...,1.370173,54,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Stage 3 Multi,S3_M05,S3_M05_raw_shift_h128_mlp_focal,raw_shift,mlp,focal,focal cell hard examples,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.550226,...,1.692307,64,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
5,Stage 3 Multi,S3_M06,S3_M06_raw_shift_h128_mlp_effective,raw_shift,mlp,focal,effective-number class balance,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.505914,...,0.809567,31,2.0,1.5,effective_number,NaN,NaN,NaN,NaN,NaN
6,Stage 3 Multi,S3_M07,S3_M07_phase_hampel_shift_mlp,phase_hampel_shift,mlp,cross_entropy,denoise multi baseline,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.537598,...,1.991614,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Stage 3 Multi,S3_M08,S3_M08_phase_hampel_shift_mlp_noise,phase_hampel_shift,mlp,cross_entropy,denoise+noise multi,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.510542,...,2.967598,42,NaN,NaN,NaN,0.01,0.3,NaN,NaN,NaN
8,Stage 3 Multi,S3_M09,S3_M09_raw_shift_mlp_l2_d03,raw_shift,mlp,cross_entropy,stacked multi,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.564151,...,1.970965,74,NaN,NaN,NaN,NaN,0.3,2.0,NaN,NaN
9,Stage 3 Multi,S3_M10,S3_M10_raw_shift_bigru_mlp,raw_shift,mlp,cross_entropy,BiGRU multi probe,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.499223,...,1.231874,44,NaN,NaN,NaN,NaN,NaN,NaN,True,64.0


,stage,run,experiment,profile,cell_head_type,cell_loss_type,purpose,status,run_dir,best_val_composite_score,val_loss,loss_weight_cell,cell_focal_gamma,cell_class_weighting,noise_sigma,dropout,num_layers,bidirectional,hidden_size
8,selection,S3_M09,S3_M09_raw_shift_mlp_l2_d03,raw_shift,mlp,cross_entropy,stacked multi,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.564151,9.021899,NaN,NaN,NaN,NaN,0.3,2.0,NaN,NaN
1,selection,S3_M02,S3_M02_top_attention_mlp,selected,mlp,cross_entropy,MLP cell head,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.559524,8.157146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,selection,S3_M05,S3_M05_raw_shift_h128_mlp_focal,raw_shift,mlp,focal,focal cell hard examples,completed,c:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...,0.550226,13.506321,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN


## Stage 4 - Final confirmation and ablation (`S4_F01`-`S4_F08`)

Mục tiêu: xác nhận top candidates bằng seed khác và ablation nhỏ. Không mở search mới ở stage này.

### S4_F01 - attention top seed43


In [33]:
selected = pick_config(top_attention_configs, 0, fallback_config(profile="raw_shift", hidden_size=96, attention_dim=32, batch_size=32, learning_rate=1e-3, weight_decay=1e-4, num_layers=1, dropout=0.2, noise_sigma=0.0, bidirectional=False))
config = train_attention.TrainConfig(experiment_name="S4_F01_attention_top_seed43")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.seed = 43
timed_train("Stage 4 Attention Final", train_attention.train, config)


Attention-GRU:  44%|████▍     | 44/100 [00:46<00:58,  1.05s/it, val_f1=0.693, val_loss=1.75]


Stage 4 Attention Final | S4_F01_attention_top_seed43 | 0.79 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S4_F01_attention_top_seed43',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.2,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'best_epoch': 30,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 45,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S4_F01_attention_top_seed43',
 'best_val_accuracy': 0.7428571428571429,
 'best_val_macro_f1': 0.699466585688649,
 'best_val_weighted_f1': 0.7443165710728736,
 'best_val_macro_precision': 0.7006870050001214,
 'best_val_macro_recall': 0.7039611057188833,
 'best_val_loss': 1.6568768892969403,
 'test_accuracy': 0.6762589928057554,
 'test_macro_f1': 0.619713355427641,
 'test_weighted_f1': 0.6779584391095181,
 'test_macro_precision': 0.6345570692044934,
 'test_macro_recall': 0.621238921360611,
 'test_loss': 2.0788408217670247,
 'elapsed_mi

### S4_F02 - attention second seed43


In [34]:
selected = pick_config(top_attention_configs, 1, fallback_config(profile="raw_shift", hidden_size=96, attention_dim=32, batch_size=32, learning_rate=1e-3, weight_decay=1e-4, num_layers=1, dropout=0.2, noise_sigma=0.0, bidirectional=False))
config = train_attention.TrainConfig(experiment_name="S4_F02_attention_second_seed43")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional"])
config.seed = 43
timed_train("Stage 4 Attention Final", train_attention.train, config)


Attention-GRU:  51%|█████     | 51/100 [01:01<00:59,  1.20s/it, val_f1=0.714, val_loss=1.93]


Stage 4 Attention Final | S4_F02_attention_second_seed43 | 1.05 min


{'phase_name': 'Attention-GRU',
 'profile': 'raw_shift',
 'experiment_name': 'S4_F02_attention_second_seed43',
 'hidden_size': 96,
 'attention_dim': 32,
 'dropout': 0.3,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 32,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'best_epoch': 37,
 'best_metric_name': 'val_macro_f1',
 'epochs_ran': 52,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\attention_gru\\S4_F02_attention_second_seed43',
 'best_val_accuracy': 0.7857142857142857,
 'best_val_macro_f1': 0.7343257248967608,
 'best_val_weighted_f1': 0.7787198454532654,
 'best_val_macro_precision': 0.7593437209375641,
 'best_val_macro_recall': 0.7293627557405941,
 'best_val_loss': 1.8822649904659816,
 'test_accuracy': 0.7122302158273381,
 'test_macro_f1': 0.6621387002092085,
 'test_weighted_f1': 0.7146179859359201,
 'test_macro_precision': 0.6640039791435568,
 'test_macro_recall': 0.6658575682583603,
 'test_loss': 2.097083669772251,
 'ela

### S4_F03 - multi top seed43


In [35]:
selected = pick_config(top_multi_configs, 0, fallback_config(profile="raw_shift", hidden_size=128, attention_dim=32, batch_size=32, learning_rate=5e-4, weight_decay=1e-4, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False, cell_head_type="mlp", cell_loss_type="cross_entropy", cell_focal_gamma=0.0, cell_label_smoothing=0.0, cell_class_weighting="inverse_frequency", loss_weight_cell=2.0))
config = train_multi.TrainConfig(experiment_name="S4_F03_multi_top_seed43")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional", "cell_head_type", "cell_loss_type", "cell_focal_gamma", "cell_label_smoothing", "cell_class_weighting", "loss_weight_cell"])
config.seed = 43

timed_train("Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  43%|████▎     | 43/100 [01:13<01:37,  1.70s/it, loss=9.3, score=0.537] 


Stage 4 Multi Final | S4_F03_multi_top_seed43 | 1.25 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S4_F03_multi_top_seed43',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.0,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 29,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5505449056809086,
 'epochs_ran': 44,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S4_F03_multi_top_seed43',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.8857142857142857,
 'v

### S4_F04 - multi top seed44


In [36]:
selected = pick_config(top_multi_configs, 0, fallback_config(profile="raw_shift", hidden_size=128, attention_dim=32, batch_size=32, learning_rate=5e-4, weight_decay=1e-4, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False, cell_head_type="mlp", cell_loss_type="cross_entropy", cell_focal_gamma=0.0, cell_label_smoothing=0.0, cell_class_weighting="inverse_frequency", loss_weight_cell=2.0))
config = train_multi.TrainConfig(experiment_name="S4_F04_multi_top_seed44")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional", "cell_head_type", "cell_loss_type", "cell_focal_gamma", "cell_label_smoothing", "cell_class_weighting", "loss_weight_cell"])
config.seed = 44

timed_train("Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  31%|███       | 31/100 [00:50<01:52,  1.63s/it, loss=7.73, score=0.508]


Stage 4 Multi Final | S4_F04_multi_top_seed44 | 0.87 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S4_F04_multi_top_seed44',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.0,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 17,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5653010498029173,
 'epochs_ran': 32,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S4_F04_multi_top_seed44',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.8428571428571429,
 'v

### S4_F05 - multi second seed43


In [37]:
selected = pick_config(top_multi_configs, 1, fallback_config(profile="raw_shift", hidden_size=128, attention_dim=32, batch_size=32, learning_rate=5e-4, weight_decay=1e-4, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False, cell_head_type="mlp", cell_loss_type="cross_entropy", cell_focal_gamma=0.0, cell_label_smoothing=0.0, cell_class_weighting="inverse_frequency", loss_weight_cell=2.0))
config = train_multi.TrainConfig(experiment_name="S4_F05_multi_second_seed43")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional", "cell_head_type", "cell_loss_type", "cell_focal_gamma", "cell_label_smoothing", "cell_class_weighting", "loss_weight_cell"])
config.seed = 43

timed_train("Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  44%|████▍     | 44/100 [01:05<01:23,  1.49s/it, loss=11.5, score=0.548]


Stage 4 Multi Final | S4_F05_multi_second_seed43 | 1.12 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S4_F05_multi_second_seed43',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.2,
 'batch_size': 32,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.0,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 30,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5679054766860232,
 'epochs_ran': 45,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S4_F05_multi_second_seed43',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.9071428571428571

### S4_F06 - multi linear head ablation


In [38]:
selected = pick_config(top_multi_configs, 0, fallback_config(profile="raw_shift", hidden_size=128, attention_dim=32, batch_size=32, learning_rate=5e-4, weight_decay=1e-4, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False, cell_head_type="mlp", cell_loss_type="cross_entropy", cell_focal_gamma=0.0, cell_label_smoothing=0.0, cell_class_weighting="inverse_frequency", loss_weight_cell=2.0))
config = train_multi.TrainConfig(experiment_name="S4_F06_multi_linear_ablation")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional", "cell_head_type", "cell_loss_type", "cell_focal_gamma", "cell_label_smoothing", "cell_class_weighting", "loss_weight_cell"])
config.seed = 43
config.cell_head_type = 'linear'
config.cell_head_dropout = 0.0
timed_train("Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  56%|█████▌    | 56/100 [01:27<01:08,  1.56s/it, loss=7.87, score=0.548]
                                

Stage 4 Multi Final | S4_F06_multi_linear_ablation | 1.48 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S4_F06_multi_linear_ablation',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'cell_head_type': 'linear',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.0,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 42,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5682439108204228,
 'epochs_ran': 57,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S4_F06_multi_linear_ablation',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.90714285

### S4_F07 - multi CE loss ablation


In [39]:
selected = pick_config(top_multi_configs, 0, fallback_config(profile="raw_shift", hidden_size=128, attention_dim=32, batch_size=32, learning_rate=5e-4, weight_decay=1e-4, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False, cell_head_type="mlp", cell_loss_type="cross_entropy", cell_focal_gamma=0.0, cell_label_smoothing=0.0, cell_class_weighting="inverse_frequency", loss_weight_cell=2.0))
config = train_multi.TrainConfig(experiment_name="S4_F07_multi_ce_ablation")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional", "cell_head_type", "cell_loss_type", "cell_focal_gamma", "cell_label_smoothing", "cell_class_weighting", "loss_weight_cell"])
config.seed = 43
config.cell_loss_type = 'cross_entropy'
config.cell_focal_gamma = 0.0
config.cell_class_weighting = 'inverse_frequency'
timed_train("Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  43%|████▎     | 43/100 [01:11<01:34,  1.65s/it, loss=9.3, score=0.537] 


Stage 4 Multi Final | S4_F07_multi_ce_ablation | 1.21 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S4_F07_multi_ce_ablation',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 2,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.0,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 29,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5505449056809086,
 'epochs_ran': 44,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S4_F07_multi_ce_ablation',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.8857142857142857,
 

### S4_F08 - multi layer ablation


In [40]:
selected = pick_config(top_multi_configs, 0, fallback_config(profile="raw_shift", hidden_size=128, attention_dim=32, batch_size=32, learning_rate=5e-4, weight_decay=1e-4, num_layers=1, dropout=0.3, noise_sigma=0.0, bidirectional=False, cell_head_type="mlp", cell_loss_type="cross_entropy", cell_focal_gamma=0.0, cell_label_smoothing=0.0, cell_class_weighting="inverse_frequency", loss_weight_cell=2.0))
config = train_multi.TrainConfig(experiment_name="S4_F08_multi_layer_ablation")
apply_common(config, selected, ["profile", "hidden_size", "attention_dim", "batch_size", "learning_rate", "weight_decay", "num_layers", "dropout", "noise_sigma", "bidirectional", "cell_head_type", "cell_loss_type", "cell_focal_gamma", "cell_label_smoothing", "cell_class_weighting", "loss_weight_cell"])
config.seed = 43
config.num_layers = 1
config.bidirectional = False
timed_train("Stage 4 Multi Final", train_multi.train, config, use_selector=False)


Multi-output Attention-GRU:  53%|█████▎    | 53/100 [01:46<01:34,  2.01s/it, loss=9.49, score=0.531]


Stage 4 Multi Final | S4_F08_multi_layer_ablation | 1.80 min


{'phase_name': 'Multi-output Attention-GRU',
 'experiment_name': 'S4_F08_multi_layer_ablation',
 'profile': 'raw_shift',
 'hidden_size': 128,
 'attention_dim': 32,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'noise_sigma': 0.0,
 'num_layers': 1,
 'bidirectional': False,
 'cell_head_type': 'mlp',
 'cell_head_hidden_size': None,
 'cell_head_dropout': 0.0,
 'cell_loss_type': 'cross_entropy',
 'cell_label_smoothing': 0.0,
 'cell_focal_gamma': 0.0,
 'cell_class_weighting': 'inverse_frequency',
 'cell_class_balance_beta': 0.9999,
 'best_epoch': 39,
 'best_metric_name': 'val_composite_score',
 'best_val_composite_score': 0.5532146202008116,
 'epochs_ran': 54,
 'run_dir': 'C:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 2 mô hình\\runs\\multi_output_best_model\\S4_F08_multi_layer_ablation',
 'selection': {'selected': False,
  'reason': 'selector disabled by caller',
  'source_run_dir': None,
  'selected_config': {}},
 'val_presence_accuracy': 0.8928571428571

## Final dashboard

Dashboard chỉ đọc planned `S1_/S2_/S3_/S4_` runs. Test metrics được hiển thị để báo cáo, không dùng cho selector.

In [41]:
overall = show_status()
display(overall)

summary = overall.groupby(["stage", "status"], dropna=False).size().reset_index(name="count")
display(summary)

if RUN_LOG:
    display(pd.DataFrame(RUN_LOG))

print("Expected output after Run All:")
print("- 35 planned training runs with config.json, metrics.json, history.csv, best_model.pt per completed run")
print("- Stage rankings for GRU, Attention-GRU, Multi-output, and final confirmation")
print("- Final candidates reported by validation metrics; test metrics used only for reporting")


,stage,run,experiment,profile,hidden_size,num_layers,dropout,batch_size,lr,purpose,...,bidirectional,cell_head_type,cell_loss_type,best_val_composite_score,val_cell_masked_macro_f1,test_cell_masked_macro_f1,test_pose_macro_f1,loss_weight_cell,cell_focal_gamma,cell_class_weighting
0,Stage 1 GRU,S1_G01,S1_G01_raw_h64,raw,64,1,0.0,32.0,0.001,raw baseline đối chứng,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Stage 1 GRU,S1_G02,S1_G02_raw_shift_h64_anchor,raw_shift,64,1,0.0,32.0,0.001,anchor ổn định từ G02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Stage 1 GRU,S1_G03,S1_G03_raw_shift_h96,raw_shift,96,1,0.0,32.0,0.001,capacity trung bình,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Stage 1 GRU,S1_G04,S1_G04_raw_shift_h128,raw_shift,128,1,0.0,32.0,0.001,capacity cao kiểm tra overfit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Stage 1 GRU,S1_G05,S1_G05_raw_shift_h64_l2_d03,raw_shift,64,2,0.3,32.0,0.001,stacked GRU nhỏ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Stage 1 GRU,S1_G06,S1_G06_raw_shift_h96_l2_d03,raw_shift,96,2,0.3,32.0,0.001,stacked GRU trung bình,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Stage 1 GRU,S1_G07,S1_G07_raw_shift_h96_b64_lr0005,raw_shift,96,1,0.0,64.0,0.0005,batch/lr ổn định,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Stage 2 Attention,S2_A01,S2_A01_top_gru_a32,selected,selected,selected,0.2,NaN,selected,top GRU -> attention a32,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Stage 2 Attention,S2_A02,S2_A02_top_gru_a48,selected,selected,selected,0.2,NaN,selected,top GRU -> attention a48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Stage 2 Attention,S2_A03,S2_A03_second_gru_a32,selected,selected,selected,0.2,NaN,selected,second GRU branch,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,stage,status,count
0,Stage 1 GRU,completed,7
1,Stage 2 Attention,completed,10
2,Stage 3 Multi,completed,10
3,Stage 4 Attention Final,completed,2
4,Stage 4 Multi Final,completed,6


,stage,experiment_name,elapsed_minutes,run_dir
0,Stage 1 GRU,S1_G01_raw_h64,0.232400,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
1,Stage 1 GRU,S1_G02_raw_shift_h64_anchor,0.593382,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
2,Stage 1 GRU,S1_G03_raw_shift_h96,0.427393,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
3,Stage 1 GRU,S1_G04_raw_shift_h128,0.824132,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
4,Stage 1 GRU,S1_G05_raw_shift_h64_l2_d03,0.730707,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
5,Stage 1 GRU,S1_G06_raw_shift_h96_l2_d03,0.491582,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
6,Stage 1 GRU,S1_G07_raw_shift_h96_b64_lr0005,0.406869,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
7,Stage 2 Attention,S2_A01_top_gru_a32,0.577504,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
8,Stage 2 Attention,S2_A02_top_gru_a48,0.622833,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...
9,Stage 2 Attention,S2_A03_second_gru_a32,0.598936,C:\Users\nguyen\Desktop\DL_V2\Phần 2 mô hình\r...


Expected output after Run All:
- 35 planned training runs with config.json, metrics.json, history.csv, best_model.pt per completed run
- Stage rankings for GRU, Attention-GRU, Multi-output, and final confirmation
- Final candidates reported by validation metrics; test metrics used only for reporting


## Reporting-only relaxed localization metrics

These cells analyze existing multi-output `predictions.csv` files after the dashboard. They do not retrain models, do not update per-run artifacts, and do not affect validation selection.


In [43]:
def relaxed_localization_metrics(frame: pd.DataFrame) -> dict[str, float]:
    cell_labels = pd.to_numeric(frame["cell_label_human"], errors="coerce")
    cell_preds = pd.to_numeric(frame["cell_pred"], errors="coerce")
    valid = frame[cell_labels.ge(0) & cell_preds.notna()].copy()
    if valid.empty:
        return {
            "cell_masked_support": 0.0,
            "cell_masked_accuracy": 0.0,
            "cell_relaxed_soft_score": 0.0,
            "cell_within_1cell": 0.0,
            "cell_within_2cell": 0.0,
            "cell_grid_l1_mean": 0.0,
            "cell_grid_linf_mean": 0.0,
            "center_mean_error_m": 0.0,
        }
    labels = valid["cell_label_human"].astype(int)
    preds = valid["cell_pred"].astype(int)
    row_distance = (labels // 5 - preds // 5).abs()
    col_distance = (labels % 5 - preds % 5).abs()
    l1_distance = row_distance + col_distance
    linf_distance = pd.concat([row_distance, col_distance], axis=1).max(axis=1)
    exact = linf_distance.eq(0)
    orthogonal = l1_distance.eq(1)
    diagonal = row_distance.eq(1) & col_distance.eq(1)
    soft_score = exact.astype(float) + orthogonal.astype(float) * 0.4 + diagonal.astype(float) * 0.3
    center_rows = frame[pd.to_numeric(frame["presence_label"], errors="coerce").eq(1)]
    center_mean_error_m = 0.0 if center_rows.empty else float(pd.to_numeric(center_rows["center_error_m"], errors="coerce").mean())
    return {
        "cell_masked_support": float(len(valid)),
        "cell_masked_accuracy": float(exact.mean()),
        "cell_relaxed_soft_score": float(soft_score.mean()),
        "cell_within_1cell": float(linf_distance.le(1).mean()),
        "cell_within_2cell": float(linf_distance.le(2).mean()),
        "cell_grid_l1_mean": float(l1_distance.mean()),
        "cell_grid_linf_mean": float(linf_distance.mean()),
        "center_mean_error_m": center_mean_error_m,
    }

summary_rows = []
for predictions_path in sorted(MULTI_RUNS_ROOT.glob("*/predictions.csv")):
    predictions = pd.read_csv(predictions_path)
    for split, split_frame in predictions.groupby("split", sort=True):
        row = {"experiment_name": predictions_path.parent.name, "split": split}
        row.update(relaxed_localization_metrics(split_frame))
        summary_rows.append(row)

relaxed_summary = pd.DataFrame(summary_rows)
summary_path = MULTI_RUNS_ROOT / "relaxed_localization_summary.csv"
relaxed_summary.to_csv(summary_path, index=False)

if relaxed_summary.empty:
    print("No multi-output predictions.csv files found.")
else:
    ranking = relaxed_summary[relaxed_summary["split"].eq("test")].sort_values(
        ["cell_relaxed_soft_score", "cell_within_1cell", "cell_masked_accuracy"],
        ascending=[False, False, False],
    )
    display(ranking)
    display(relaxed_summary.sort_values(["experiment_name", "split"]))

print(f"Wrote reporting-only relaxed localization summary: {summary_path.name}")


,experiment_name,split,cell_masked_support,cell_masked_accuracy,cell_relaxed_soft_score,cell_within_1cell,cell_within_2cell,cell_grid_l1_mean,cell_grid_linf_mean,center_mean_error_m
26,S4_F06_multi_linear_ablation,test,130.0,0.269231,0.419231,0.669231,0.861538,1.561538,1.238462,0.724590
20,S4_F03_multi_top_seed43,test,130.0,0.261538,0.396923,0.623077,0.884615,1.592308,1.261538,0.720928
28,S4_F07_multi_ce_ablation,test,130.0,0.261538,0.396923,0.623077,0.884615,1.592308,1.261538,0.720928
24,S4_F05_multi_second_seed43,test,130.0,0.246154,0.395385,0.638462,0.876923,1.561538,1.261538,0.704989
16,S3_M09_raw_shift_mlp_l2_d03,test,130.0,0.230769,0.384615,0.638462,0.846154,1.692308,1.315385,0.697802
2,S3_M02_top_attention_mlp,test,130.0,0.238462,0.378462,0.607692,0.892308,1.646154,1.261538,0.752742
12,S3_M07_phase_hampel_shift_mlp,test,130.0,0.238462,0.374615,0.600000,0.884615,1.715385,1.323077,0.741046
30,S4_F08_multi_layer_ablation,test,130.0,0.207692,0.340769,0.561538,0.861538,1.823077,1.430769,0.717125
8,S3_M05_raw_shift_h128_mlp_focal,test,130.0,0.207692,0.326923,0.530769,0.838462,1.869231,1.469231,0.727401
6,S3_M04_raw_shift_h128_mlp_cell2,test,130.0,0.184615,0.326154,0.569231,0.869231,1.838462,1.423077,0.749302


,experiment_name,split,cell_masked_support,cell_masked_accuracy,cell_relaxed_soft_score,cell_within_1cell,cell_within_2cell,cell_grid_l1_mean,cell_grid_linf_mean,center_mean_error_m
0,S3_M01_top_attention_linear,test,130.0,0.192308,0.323846,0.546154,0.784615,1.892308,1.530769,0.740144
1,S3_M01_top_attention_linear,val,132.0,0.257576,0.357576,0.522727,0.734848,2.060606,1.537879,0.798454
2,S3_M02_top_attention_mlp,test,130.0,0.238462,0.378462,0.607692,0.892308,1.646154,1.261538,0.752742
3,S3_M02_top_attention_mlp,val,132.0,0.257576,0.405303,0.651515,0.856061,1.583333,1.250000,0.787494
4,S3_M03_raw_shift_h128_mlp,test,130.0,0.138462,0.265385,0.476923,0.746154,2.176923,1.723077,0.780982
5,S3_M03_raw_shift_h128_mlp,val,132.0,0.174242,0.328030,0.590909,0.803030,1.969697,1.462121,0.811975
6,S3_M04_raw_shift_h128_mlp_cell2,test,130.0,0.184615,0.326154,0.569231,0.869231,1.838462,1.423077,0.749302
7,S3_M04_raw_shift_h128_mlp_cell2,val,132.0,0.250000,0.373485,0.583333,0.871212,1.734848,1.333333,0.759306
8,S3_M05_raw_shift_h128_mlp_focal,test,130.0,0.207692,0.326923,0.530769,0.838462,1.869231,1.469231,0.727401
9,S3_M05_raw_shift_h128_mlp_focal,val,132.0,0.318182,0.439394,0.643939,0.886364,1.553030,1.181818,0.763067


Wrote reporting-only relaxed localization summary: relaxed_localization_summary.csv
